In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator


In [2]:
#Setting up the Registers and Circuit
data = QuantumRegister(3, name = "data")  # 3 qubits for the bit flip code
ancilla = QuantumRegister(2, name = "ancilla")  # 2 ancilla qubits for syndrome measurement
syndrome = ClassicalRegister(2, name = "syndrome")  # 2 classical bits to store the syndrome measurement results

qc = QuantumCircuit(data, ancilla, syndrome) # Create the quantum circuit with the defined registers


In [3]:
#Encoding Stage
#Here will will entangle data[0] with data[1] and data[2] to create the logical qubit state
qc.cx(data[0], data[1])  # CNOT from data[0] to data[1]
qc.cx(data[0], data[2])  # CNOT from data[0] to data[2]
qc.barrier # Add a barrier for clarity in the circuit diagram and also to avaid any unwanted optimizations during transpilation

<bound method QuantumCircuit.barrier of <qiskit.circuit.quantumcircuit.QuantumCircuit object at 0x00000201DD2A3190>>

In [4]:
qc.draw()

data_0: ──■────■──
            ┌─┴─┐  │  
    data_1: ┤ X ├──┼──
            └───┘┌─┴─┐
    data_2: ─────┤ X ├
                 └───┘
 ancilla_0: ──────────
                      
 ancilla_1: ──────────
                      
syndrome: 2/══════════

In [5]:
#ERROR INJECTION STAGE
# Let's pretend cosmic radiation flipped the first data qubit
print("Injecting an error on Data Qubit 0...")
qc.x(data[2]) 
qc.barrier()

Injecting an error on Data Qubit 0...


CircuitInstruction(operation=Instruction(name='barrier', num_qubits=5, num_clbits=0, params=[]), qubits=(<Qubit register=(3, "data"), index=0>, <Qubit register=(3, "data"), index=1>, <Qubit register=(3, "data"), index=2>, <Qubit register=(2, "ancilla"), index=0>, <Qubit register=(2, "ancilla"), index=1>), clbits=())

In [6]:
# 4. SYNDROME EXTRACTION STAGE
# Check if data[0] and data[1] are different (store result in ancilla[0])
qc.cx(data[0], ancilla[0])
qc.cx(data[1], ancilla[0])

In [7]:
qc.draw()

░           
    data_0: ──■────■────────░───■───────
            ┌─┴─┐  │        ░   │       
    data_1: ┤ X ├──┼────────░───┼────■──
            └───┘┌─┴─┐┌───┐ ░   │    │  
    data_2: ─────┤ X ├┤ X ├─░───┼────┼──
                 └───┘└───┘ ░ ┌─┴─┐┌─┴─┐
 ancilla_0: ────────────────░─┤ X ├┤ X ├
                            ░ └───┘└───┘
 ancilla_1: ────────────────░───────────
                            ░           
syndrome: 2/════════════════════════════

In [8]:
# Check if data[1] and data[2] are different (store result in ancilla[1])
qc.cx(data[1], ancilla[1])
qc.cx(data[2], ancilla[1])
qc.barrier()
qc.draw()

░                      ░ 
    data_0: ──■────■────────░───■──────────────────░─
            ┌─┴─┐  │        ░   │                  ░ 
    data_1: ┤ X ├──┼────────░───┼────■────■────────░─
            └───┘┌─┴─┐┌───┐ ░   │    │    │        ░ 
    data_2: ─────┤ X ├┤ X ├─░───┼────┼────┼────■───░─
                 └───┘└───┘ ░ ┌─┴─┐┌─┴─┐  │    │   ░ 
 ancilla_0: ────────────────░─┤ X ├┤ X ├──┼────┼───░─
                            ░ └───┘└───┘┌─┴─┐┌─┴─┐ ░ 
 ancilla_1: ────────────────░───────────┤ X ├┤ X ├─░─
                            ░           └───┘└───┘ ░ 
syndrome: 2/═════════════════════════════════════════

We are facing an Issue above. Qiskit is adding the CNOT gates multiple times as we are running the cell. 

In [9]:
qc.draw(output='text') # 'mpl' makes it look like a nice graphical diagram

░                      ░ 
    data_0: ──■────■────────░───■──────────────────░─
            ┌─┴─┐  │        ░   │                  ░ 
    data_1: ┤ X ├──┼────────░───┼────■────■────────░─
            └───┘┌─┴─┐┌───┐ ░   │    │    │        ░ 
    data_2: ─────┤ X ├┤ X ├─░───┼────┼────┼────■───░─
                 └───┘└───┘ ░ ┌─┴─┐┌─┴─┐  │    │   ░ 
 ancilla_0: ────────────────░─┤ X ├┤ X ├──┼────┼───░─
                            ░ └───┘└───┘┌─┴─┐┌─┴─┐ ░ 
 ancilla_1: ────────────────░───────────┤ X ├┤ X ├─░─
                            ░           └───┘└───┘ ░ 
syndrome: 2/═════════════════════════════════════════

In [10]:
%pip install pylatexenc matplotlib

Note: you may need to restart the kernel to use updated packages.


In [11]:
# 5. MEASUREMENT STAGE
# Measure the ancilla qubits to get our syndrome (features for our future ML model!)
qc.measure(ancilla[0], syndrome[0])
qc.measure(ancilla[1], syndrome[1])

In [12]:
# 6. RUN THE SIMULATION
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)
job = simulator.run(compiled_circuit, shots=10) # Run it 10 times
result = job.result()
counts = result.get_counts()

In [13]:
print("\n--- MEASUREMENT RESULTS ---")
print("Format: 'Syndrome_1 Syndrome_0' : Number of times observed")
print(counts)


--- MEASUREMENT RESULTS ---
Format: 'Syndrome_1 Syndrome_0' : Number of times observed
{'10': 10}
